In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### Image Classification Neural Network
Using the Nepali Numbers and Letters Dataset, build and evaluate a neural network for multi-class image classification.

Dataset: Nepali Numbers and Letters Dataset (linked in the original question).

The notebook covers:

* Data preparation: inspect class folders, count and preview images, remove corrupted files, resize images, normalize pixel values, and split the data into training, validation, and test sets.
* Neural network: build a convolutional neural network (CNN) for multi-class classification.
* Hyperparameter tuning: compare a small set of filter/dropout configurations using validation performance.
* Evaluation: report accuracy, macro/weighted precision, recall and F1-score, a classification report, and a confusion matrix.
* Performance analysis: inspect training/validation curves and discuss model limitations.

Deliverable: reproducible code and an evidence-based report using the metrics produced when the notebook is run.

#### Disclaimer: The Assignment is not supported in my laptop.

In [ ]:
print("Using TensorFlow Version:", tf.__version__)

In [ ]:
# Set random seed for reproducibility
tf.keras.utils.set_random_seed(42)

In [ ]:
BATCH_SIZE = 32
IMG_HEIGHT = 32
IMG_WIDTH = 32

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    'data/train',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE
)

In [ ]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    'data/train',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    color_mode='grayscale',
    batch_size=BATCH_SIZE
)

In [ ]:
test_ds = tf.keras.utils.image_dataset_from_directory(
    'data/test',
    shuffle=False,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    color_mode='grayscale',
    batch_size=BATCH_SIZE
)

In [ ]:
# Extract class names
class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Total Classes Found: {num_classes}")
print(f"Class Names: {class_names}")

In [ ]:
# Normalization Layer: Rescale pixel values from [0, 255] to [0.0, 1.0]
normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

In [ ]:
# Optimize dataset performance using buffered prefetching
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().buffered_prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().buffered_prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().buffered_prefetch(buffer_size=AUTOTUNE)

In [ ]:
model = models.Sequential([
    # First Convolutional Block
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 1)),
    layers.MaxPooling2D((2, 2)),
    
    # Second Convolutional Block
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Third Convolutional Block
    layers.Conv2D(64, (3, 3), activation='relu'),
    
    # Flatten & Dense Layers for Classification
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5), # Regularization to prevent overfitting
    layers.Dense(num_classes, activation='softmax') # Output layer for multi-class classification
])

In [ ]:
# Compile the Model
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

model.summary()

In [ ]:
# 3. HYPERPARAMETER TUNING & TRAINING
print("\n--- Training Model ---")
EPOCHS = 15

# Callbacks for early stopping if validation loss stops improving
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
#EVALUATION
print("\n--- Evaluating Model on Test Set ---")

# Collect true labels and predictions
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

In [ ]:
accuracy = accuracy_score(y_true, y_pred)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

print(f"\nFinal Test Accuracy: {accuracy * 100:.2f}%")
print(f"Macro Precision: {precision_macro:.4f} | Macro Recall: {recall_macro:.4f} | Macro F1-Score: {f1_macro:.4f}")
print(f"Weighted Precision: {precision_weighted:.4f} | Weighted Recall: {recall_weighted:.4f} | Weighted F1-Score: {f1_weighted:.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', cbar=True)
plt.title('Confusion Matrix - Nepali Numbers & Letters Classifier')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()